# Generate Daily Modeling Dataset

**Prerequisites**
1. `GEE_local_HSA_Daily_Climate.ipynb` completed and files downloaded to
   `{OUT_DIR}/DRIVE_CLIMATE_BY_HSA_DOWNLOAD_DAILY_{VERSION}/`
2. `Population_Allocation_Probabilistic_v2.ipynb` run for the chosen `BOUNDARY_VERSION`
3. `data/INF_patient_visits.csv` present (real data) or `data/SYNMODINF_patient_visits.csv` (synthetic)

**Outputs**
- `{OUT_DIR}/INF_footprint_daily_diarrheal_{VERSION}.csv` — daily case counts per HSA
- `{OUT_DIR}/modeling/INF_footprint_daily_modeling_dataset_{VERSION}.csv` — merged panel with 14-day lags


In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────
# Choose which HSA boundary bundle to use. Must match a bundle from HSA_FINAL.ipynb.
#   'v6'  — original greedy algorithm (no post-selection corrections)
#   'v7'  — + anchor upgrade/demotion + major-orphan promotion
#   'v8'  — + satellite bubble boundaries
BOUNDARY_VERSION = "v7"         # change as needed

NETWORK          = "INF"        # change as needed (INF or NCD)
HSA_MODE         = "footprint"  # change as needed

# Study period — dates outside this window are excluded from the disease count.
# Codes changed mid-2022; June 2-30 2022 is a system gap (flagged, not dropped here).
STUDY_START = "2022-07-01"      # first valid date (after diagnostic code transition)
STUDY_END   = "2024-01-31"      # last date of study window
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
from pathlib import Path
import os
import subprocess, sys

BASE_DIR = Path(".")
OUT_DIR = Path(os.environ.get("HSA_OUT_DIR", os.environ.get("PIPELINE_OUT_DIR", "out")))
MODELING_DIR = OUT_DIR / "modeling"
MODELING_DIR.mkdir(parents=True, exist_ok=True)

print("Working directory:", BASE_DIR.resolve())
print("Output directory: ", OUT_DIR)
print("Boundary version: ", BOUNDARY_VERSION)
print("Network:          ", NETWORK)
print("Study window:     ", STUDY_START, "to", STUDY_END)

# --- Prerequisite check ---
required = {
    "HSA boundaries":       OUT_DIR / f"{NETWORK}_{HSA_MODE}_hsas_{BOUNDARY_VERSION}.geojson",
    "Facility assignments": OUT_DIR / f"{NETWORK}_{HSA_MODE}_facility_hsa_assignments_{BOUNDARY_VERSION}.csv",
    "Daily climate dir":    OUT_DIR / f"DRIVE_CLIMATE_BY_HSA_DOWNLOAD_DAILY_{BOUNDARY_VERSION.upper()}",
}
patient_candidates = [
    Path(f"data/{NETWORK}_patient_visits.csv"),
    Path(f"data/SYNMOD{NETWORK}_patient_visits.csv"),
]
patient_file = next((p for p in patient_candidates if p.exists()), None)
required["Patient visits"] = patient_file if patient_file else Path(f"data/{NETWORK}_patient_visits.csv")

print("\nChecking prerequisites...")
all_ok = True
for name, path in required.items():
    ok = path.exists()
    print(f"  {'OK' if ok else 'MISSING':<8} {name}: {path}")
    if not ok:
        all_ok = False

if not all_ok:
    raise FileNotFoundError("Missing prerequisites listed above. Complete them before continuing.")

print("\nAll prerequisites satisfied.")


In [ ]:
# STEP 1 — Daily disease counts
print("=" * 60)
print("STEP 1: Daily diarrheal counts")
print("=" * 60)

result = subprocess.run(
    [sys.executable, "generate_daily_disease_counts.py",
     "--network",          NETWORK,
     "--hsa-mode",         HSA_MODE,
     "--study-start",      STUDY_START,
     "--study-end",        STUDY_END,
     "--out-dir",          str(OUT_DIR),
     "--boundary-version", BOUNDARY_VERSION],
    capture_output=False,
)
if result.returncode != 0:
    raise RuntimeError("generate_daily_disease_counts.py failed")
print("Done.")


In [ ]:
# STEP 2 — Assemble modeling dataset
print("=" * 60)
print("STEP 2: Assemble daily modeling dataset")
print("=" * 60)

result = subprocess.run(
    [sys.executable, "prepare_daily_modeling_dataset.py",
     "--network",          NETWORK,
     "--hsa-mode",         HSA_MODE,
     "--boundary-version", BOUNDARY_VERSION,
     "--out-dir",          str(OUT_DIR)],
    capture_output=False,
)
if result.returncode != 0:
    raise RuntimeError("prepare_daily_modeling_dataset.py failed")
print("Done.")


In [ ]:
# STEP 3 — Verify output
import pandas as pd

out_file = MODELING_DIR / f"{NETWORK}_{HSA_MODE}_daily_modeling_dataset_{BOUNDARY_VERSION}.csv"
df = pd.read_csv(out_file, parse_dates=["date"])

print(f"Output:      {out_file.name}")
print(f"Shape:       {df.shape}")
print(f"HSAs:        {df['hsa_id'].nunique()}")
print(f"Date range:  {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Columns:     {list(df.columns[:12])} ...")
print()
print("Non-zero outcome days:", (df["diarrheal_count"] > 0).sum())
print("Zero-count fraction:  ", f"{(df['diarrheal_count']==0).mean()*100:.1f}%")
print()
print("Climate coverage (non-null P_precip):",
      f"{df['P_precip'].notna().sum():,} / {len(df):,}")
print()
lag_cols = [c for c in df.columns if "lag" in c][:8]
print("Lag columns present (sample):", lag_cols)


In [ ]:
# AUTO_NOTEBOOK_SUMMARY_V1
from pathlib import Path
from datetime import datetime
import os
import json

NOTEBOOK_NAME = "Generate_Daily_Modeling_Dataset"
NETWORK = globals().get('NETWORK', os.environ.get('NETWORK', 'INF'))
HSA_MODE = globals().get('HSA_MODE', os.environ.get('HSA_MODE', ''))

suffix = f"{NETWORK}_{HSA_MODE}" if HSA_MODE else f"{NETWORK}"

out_root = Path(globals().get('OUT_DIR', globals().get('OUT_ROOT', os.environ.get('HSA_OUT_DIR', os.environ.get('PIPELINE_OUT_DIR', "out")))))
summary_dir = out_root / 'textresults'
summary_dir.mkdir(parents=True, exist_ok=True)
BOUNDARY_VERSION = globals().get('BOUNDARY_VERSION', os.environ.get('BOUNDARY_VERSION', os.environ.get('PIPELINE_VERSION', 'v7')))
summary_path = summary_dir / f"{NOTEBOOK_NAME}_{suffix}_{BOUNDARY_VERSION}_results.md"

files = [p for p in out_root.rglob('*') if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.md', '.txt', '.png', '.pdf', '.geojson', '.parquet'}]
files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
important = files[:120]

NL = "\n"

blocks = []
blocks.append(f"# Notebook Results: {NOTEBOOK_NAME}")

meta = [
    f"- Generated: {datetime.now().isoformat(timespec='seconds')}",
    f"- Network: {NETWORK}",
    f"- HSA mode: {HSA_MODE}",
    f"- Boundary version: {BOUNDARY_VERSION}",
]
blocks.append(NL.join(meta))

file_lines = ['## Important Output Files', '']
for p in important:
    file_lines.append(f"- `{p}`")
blocks.append(NL.join(file_lines))

nb_path = Path(f"{NOTEBOOK_NAME}.ipynb")
if nb_path.exists():
    try:
        nb_data = json.loads(nb_path.read_text())
        blocks.append('## Displayed Cell Outputs')

        for idx, cell in enumerate(nb_data.get('cells', []), start=1):
            if cell.get('cell_type') != 'code':
                continue
            outputs = cell.get('outputs', []) or []
            if not outputs:
                continue

            blocks.append(f"### Cell {idx}")

            for out in outputs:
                otype = out.get('output_type')

                if otype == 'stream':
                    text = ''.join(out.get('text', [])) if isinstance(out.get('text', []), list) else str(out.get('text', ''))
                    text = text.rstrip()
                    if text:
                        blocks.append("```text" + NL + text + NL + "```")

                elif otype in ('execute_result', 'display_data'):
                    data = out.get('data', {})
                    if 'text/markdown' in data:
                        md = ''.join(data['text/markdown']) if isinstance(data['text/markdown'], list) else str(data['text/markdown'])
                        md = md.rstrip()
                        if md:
                            blocks.append(md)
                    elif 'text/plain' in data:
                        txt = ''.join(data['text/plain']) if isinstance(data['text/plain'], list) else str(data['text/plain'])
                        txt = txt.rstrip()
                        if txt:
                            blocks.append("```text" + NL + txt + NL + "```")
                    elif 'text/html' in data:
                        html = ''.join(data['text/html']) if isinstance(data['text/html'], list) else str(data['text/html'])
                        html = html.rstrip()
                        if html:
                            blocks.append("```html" + NL + html + NL + "```")
                    elif 'image/png' in data or 'image/jpeg' in data:
                        blocks.append('_[Image output omitted in text summary]_')

                elif otype == 'error':
                    tb = out.get('traceback', []) or []
                    if tb:
                        err = NL.join(str(t) for t in tb)
                    else:
                        err = f"{out.get('ename', 'Error')}: {out.get('evalue', '')}"
                    blocks.append("```text" + NL + err + NL + "```")

    except Exception as e:
        blocks.append('## Displayed Cell Outputs')
        blocks.append(f"Could not parse notebook outputs: {e}")

summary = (NL + NL).join(b for b in blocks if b and str(b).strip()) + NL
summary_path.write_text(summary)
print(f"Saved notebook summary: {summary_path}")
